# 01. 법령 API 데이터 수집

법제처 **국가법령정보 공동활용 API**(law.go.kr DRF)로 법령·판례·법령해석례를 수집해
`data/01_raw/`에 저장한다.

| target  | 설명        | 검색 방식        | 중복 제거 기준       | 저장 형식                         |
| ------- | ----------- | ---------------- | -------------------- | --------------------------------- |
| `eflaw` | 현행법령    | 명칭검색(search=1) | `법령ID`             | 법령별 JSON `eflaw/{법령명}.json` |
| `prec`  | 판례        | 본문검색(search=2) | `판례일련번호`       | `prec.jsonl`                      |
| `expc`  | 법령해석례  | 본문검색(search=2) | `법령해석례일련번호` | `expc.jsonl`                      |

처리 흐름: **검색 → 합치기 → 1차 dedup(ID) → 상세 조회 → 2차 검증 → 저장**

> ⚠️ 2차 검증에서 본문이 비었거나 `"일치하는 …없습니다"` 인 레코드를 제거한다.
> ⚠️ `상세링크` 필드에는 OC(API 키)가 노출되므로 저장 전에 제거한다.

## 0. 공통 설정

In [1]:
import json
import math
import re
import time
from pathlib import Path
from typing import Any, Literal

import requests
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import os

load_dotenv("../.env")
OC = os.environ["LAW_API_OC"]  # .env 에 LAW_API_OC 필요

SEARCH_URL = "https://www.law.go.kr/DRF/lawSearch.do"
SERVICE_URL = "https://www.law.go.kr/DRF/lawService.do"

# 재시도(429/5xx) + 백오프 세션
_retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
_session = requests.Session()
_session.mount("https://", HTTPAdapter(max_retries=_retry))
_session.mount("http://", HTTPAdapter(max_retries=_retry))

# 저장 경로
RAW_DIR = Path("../data/01_raw")
EFLAW_DIR = RAW_DIR / "eflaw"
EFLAW_DIR.mkdir(parents=True, exist_ok=True)

print("OC:", OC)
print("저장 경로:", RAW_DIR.resolve())

OC: jungia21
저장 경로: D:\project\SKN30-3rd-4Team-dev\data\01_raw


### API 호출 헬퍼

- `api()` : 단일 호출 래퍼 (`resultCode` 검사 포함)
- `fetch_list()` : 목록 조회 + 페이지네이션 자동 처리
- `fetch_details()` : 항목별 상세 조회 + 본문 유효성 필터 + 메타 병합

In [2]:
# 검색/상세 응답의 루트·items 키가 target 마다 달라 헬퍼로 흡수한다.
_SEARCH_ROOT = {"eflaw": "LawSearch", "prec": "PrecSearch", "expc": "Expc"}
_SEARCH_ITEMS = {"eflaw": "law", "prec": "prec", "expc": "expc"}


def api(
    target: str,
    service: Literal["search", "detail"] = "search",
    params: dict[str, Any] | None = None,
    timeout: int = 60,
) -> dict[str, Any]:
    """law.go.kr DRF 단일 호출. JSON dict 반환."""
    url = SERVICE_URL if service == "detail" else SEARCH_URL
    req = {"OC": OC, "type": "JSON", "target": target, **(params or {})}
    resp = _session.get(url, params=req, timeout=timeout)
    resp.raise_for_status()
    return resp.json()


def _search_root(resp: dict, target: str) -> dict:
    return resp.get(_SEARCH_ROOT[target], {})


def _extract_items(resp: dict, target: str) -> list[dict]:
    root = _search_root(resp, target)
    items = root.get(_SEARCH_ITEMS[target], [])
    if isinstance(items, dict):  # 결과 1건이면 dict 로 옴
        items = [items]
    return items if isinstance(items, list) else []


def fetch_list(
    target: str,
    query: str,
    search: int = 1,
    max_items: int | None = None,
    extra: dict[str, Any] | None = None,
) -> list[dict]:
    """목록 조회 + 페이지네이션. search=1 명칭 / search=2 본문.
    extra: target별 추가 파라미터(예: eflaw 의 nw=3 → 현행만)."""
    display = 100
    base = {"query": query, "search": search, "display": display, **(extra or {})}

    first = api(target, "search", {**base, "page": 1})
    total = int(_search_root(first, target).get("totalCnt", 0))
    if max_items:
        total = min(total, max_items)
    pages = math.ceil(total / display)

    items = _extract_items(first, target)
    for page in range(2, pages + 1):
        items.extend(_extract_items(api(target, "search", {**base, "page": page}), target))
        time.sleep(0.3)
    return items[:max_items] if max_items else items


def _strip_link_fields(item: dict) -> None:
    """상세링크 계열 필드 제거 (OC=API키 노출 방지)."""
    for k in [k for k in item if "상세링크" in k]:
        item.pop(k, None)


def _is_invalid_body(body: Any) -> bool:
    """빈 본문 또는 '일치하는 …없습니다' 에러 응답이면 True."""
    if not body:
        return True
    s = json.dumps(body, ensure_ascii=False)
    return ("일치하는" in s and "없습니다" in s)


def fetch_details(
    target: str,
    items: list[dict],
    id_field: str,
    delay: float = 0.5,
) -> list[dict]:
    """항목별 상세 조회 → 본문 유효성 필터 → 메타+본문 병합."""
    results, dropped = [], 0
    total = len(items)
    print(f"  상세 조회 시작: 총 {total}건 (건당 ~{delay}s sleep)")
    for i, item in enumerate(items, 1):
        doc_id = str(item.get(id_field, ""))
        if not doc_id:
            dropped += 1
            continue
        body = api(target, "detail", {"ID": doc_id})
        time.sleep(delay)
        if _is_invalid_body(body):
            dropped += 1
            continue
        _strip_link_fields(item)
        results.append({**item, "본문": body})
        if i % 20 == 0 or i == total:
            print(f"    진행 {i}/{total} — 유효 {len(results)} / 제거 {dropped}", flush=True)
    print(f"  상세 조회: {len(results)}건 유효 / {dropped}건 제거(빈·에러 본문)")
    return results


def dedup_by(items: list[dict], id_field: str) -> list[dict]:
    """id_field 기준 1차 중복 제거(빈 ID 제외)."""
    seen: set[str] = set()
    out: list[dict] = []
    for item in items:
        doc_id = str(item.get(id_field, ""))
        if doc_id and doc_id not in seen:
            seen.add(doc_id)
            out.append(item)
    return out

## 1. 법령 (eflaw, 현행법령)

정확한 법령명으로 명칭검색(`search=1`, `nw=3`=현행만) → `법령명한글` 정확 일치 + `현행연혁코드=="현행"`
만 채택 → `법령ID` dedup → 상세 조회 → **법령별 개별 JSON** 으로 저장.

> ⚠️ eflaw 도 `nw` 없이 검색하면 같은 `법령ID` 의 **연혁(과거 시행본)이 함께** 반환된다.
> (예: 주택임대차보호법 → 29건) `nw=3` 으로 현행만 받아 법명당 1건이 곧 최신 현행이 되게 한다.

In [3]:
EFLAW_QUERIES: list[str] = [
    # 기본 6
    "주택임대차보호법",
    "주택임대차보호법 시행령",
    "민법",
    "부동산등기법",
    "공인중개사법",
    "민사집행법",
    # 추가 7
    "전세사기피해자 지원 및 주거안정에 관한 특별법",
    "부동산 거래신고 등에 관한 법률",
    "주민등록법",
    "상가건물 임대차보호법",
    "부동산등기규칙",
    "공인중개사법 시행규칙",
    # 확장 4
    "민간임대주택에 관한 특별법",
    "국세징수법",
    "집합건물의 소유 및 관리에 관한 법률",
    "주택도시기금법",
]


def _safe_filename(name: str) -> str:
    """법령명 → 안전한 파일명(공백·특수문자 정리)."""
    name = re.sub(r'[\\/:*?"<>|]', "", name)
    return re.sub(r"\s+", "_", name.strip())


# 1) 명칭검색(nw=3=현행만) → 정확 일치 + 현행 필터 → 법령ID dedup
#    ※ eflaw 도 nw 없이는 연혁(과거 시행본)이 함께 반환되므로 nw=3 으로 현행만 받는다.
eflaw_items: list[dict] = []
for q in EFLAW_QUERIES:
    hits = fetch_list("eflaw", query=q, search=1, extra={"nw": 3})
    exact = [
        it for it in hits
        if it.get("법령명한글", "").strip() == q and it.get("현행연혁코드") == "현행"
    ]
    print(f"[{q}] 검색 {len(hits)}건 → 정확일치·현행 {len(exact)}건")
    eflaw_items.extend(exact)
    time.sleep(0.3)

eflaw_items = dedup_by(eflaw_items, "법령ID")
print(f"\n법령ID dedup 후: {len(eflaw_items)}건")
missing = [q for q in EFLAW_QUERIES if q not in {it.get("법령명한글", "").strip() for it in eflaw_items}]
if missing:
    print("⚠️ 정확 일치 못 찾은 query(정식 명칭 확인 필요):", missing)

[주택임대차보호법] 검색 2건 → 정확일치·현행 1건
[주택임대차보호법 시행령] 검색 1건 → 정확일치·현행 1건
[민법] 검색 6건 → 정확일치·현행 1건
[부동산등기법] 검색 2건 → 정확일치·현행 1건
[공인중개사법] 검색 3건 → 정확일치·현행 1건
[민사집행법] 검색 2건 → 정확일치·현행 1건
[전세사기피해자 지원 및 주거안정에 관한 특별법] 검색 3건 → 정확일치·현행 1건
[부동산 거래신고 등에 관한 법률] 검색 3건 → 정확일치·현행 1건
[주민등록법] 검색 3건 → 정확일치·현행 1건
[상가건물 임대차보호법] 검색 2건 → 정확일치·현행 1건
[부동산등기규칙] 검색 1건 → 정확일치·현행 1건
[공인중개사법 시행규칙] 검색 1건 → 정확일치·현행 1건
[민간임대주택에 관한 특별법] 검색 3건 → 정확일치·현행 1건
[국세징수법] 검색 3건 → 정확일치·현행 1건
[집합건물의 소유 및 관리에 관한 법률] 검색 2건 → 정확일치·현행 1건
[주택도시기금법] 검색 3건 → 정확일치·현행 1건

법령ID dedup 후: 16건


In [4]:
# 2) 상세 조회(법령ID) → 본문 유효성 필터 → 2차 dedup
eflaw_details = fetch_details("eflaw", eflaw_items, id_field="법령ID")
eflaw_details = dedup_by(eflaw_details, "법령ID")

# 3) 법령별 개별 JSON 저장
for rec in eflaw_details:
    fname = _safe_filename(rec.get("법령명한글", rec.get("법령ID", "unknown")))
    path = EFLAW_DIR / f"{fname}.json"
    with path.open("w", encoding="utf-8") as f:
        json.dump(rec, f, ensure_ascii=False, indent=2)

print(f"법령 저장 완료: {len(eflaw_details)}개 파일 → {EFLAW_DIR}")
for rec in eflaw_details:
    print(f"  - {rec.get('법령명한글')} (시행 {rec.get('시행일자')}, 법령ID {rec.get('법령ID')})")

  상세 조회 시작: 총 16건 (건당 ~0.5s sleep)
    진행 16/16 — 유효 16 / 제거 0
  상세 조회: 16건 유효 / 0건 제거(빈·에러 본문)
법령 저장 완료: 16개 파일 → ..\data\01_raw\eflaw
  - 주택임대차보호법 (시행 20260102, 법령ID 001248)
  - 주택임대차보호법 시행령 (시행 20260102, 법령ID 004950)
  - 민법 (시행 20260317, 법령ID 001706)
  - 부동산등기법 (시행 20250131, 법령ID 001697)
  - 공인중개사법 (시행 20260215, 법령ID 001654)
  - 민사집행법 (시행 20260201, 법령ID 009290)
  - 전세사기피해자 지원 및 주거안정에 관한 특별법 (시행 20260512, 법령ID 014450)
  - 부동산 거래신고 등에 관한 법률 (시행 20240517, 법령ID 012480)
  - 주민등록법 (시행 20250722, 법령ID 001655)
  - 상가건물 임대차보호법 (시행 20260512, 법령ID 009276)
  - 부동산등기규칙 (시행 20250801, 법령ID 005868)
  - 공인중개사법 시행규칙 (시행 20240710, 법령ID 007292)
  - 민간임대주택에 관한 특별법 (시행 20260102, 법령ID 000243)
  - 국세징수법 (시행 20260602, 법령ID 001585)
  - 집합건물의 소유 및 관리에 관한 법률 (시행 20230929, 법령ID 001262)
  - 주택도시기금법 (시행 20260602, 법령ID 012226)


## 2. 판례 (prec)

쟁점 키워드로 본문검색(`search=2`) → `판례일련번호` 1차 dedup → 상세 조회 →
**2차 검증(본문 유효성 + ID 유일성)** → `prec.jsonl` 저장.
단일어는 노이즈가 커서 2어절 쟁점 구 위주로 구성.

In [5]:
PREC_QUERIES: list[str] = [
    # 보증금 보호·권리
    "임대차보증금", "보증금 반환", "대항력", "우선변제권", "최우선변제",
    "소액임차인", "확정일자", "임차권등기명령", "전입신고",
    # 갱신·종료
    "계약갱신청구권", "묵시적 갱신", "갱신거절", "임대차 해지", "차임 연체", "차임 증감",
    # 전세사기·권리침해
    "전세사기", "깡통전세", "가장임대차", "무권대리", "이중임대차", "사해행위", "명의신탁",
    # 경매·배당
    "임의경매", "강제경매", "배당요구", "배당이의", "인도명령", "건물명도",
    # 수선·원상회복
    "임대인 수선의무", "원상회복", "누수", "하자", "통상의 손모",
    # 중개
    "공인중개사 책임", "중개대상물 확인설명", "중개보수",
]

MAX_PER_QUERY = 300  # query 당 안전 상한

# 1) 본문검색 → 합치기 → 판례일련번호 1차 dedup
prec_items: list[dict] = []
for q in PREC_QUERIES:
    hits = fetch_list("prec", query=q, search=2, max_items=MAX_PER_QUERY)
    print(f"[{q}] {len(hits)}건")
    prec_items.extend(hits)
    time.sleep(0.3)

before = len(prec_items)
prec_items = dedup_by(prec_items, "판례일련번호")
print(f"\n수집 {before}건 → 판례일련번호 1차 dedup {len(prec_items)}건")

[임대차보증금] 300건
[보증금 반환] 300건
[대항력] 300건
[우선변제권] 300건
[최우선변제] 233건
[소액임차인] 300건
[확정일자] 300건
[임차권등기명령] 109건
[전입신고] 300건
[계약갱신청구권] 298건
[묵시적 갱신] 286건
[갱신거절] 300건
[임대차 해지] 300건
[차임 연체] 300건
[차임 증감] 54건
[전세사기] 109건
[깡통전세] 0건
[가장임대차] 300건
[무권대리] 300건
[이중임대차] 300건
[사해행위] 300건
[명의신탁] 300건
[임의경매] 300건
[강제경매] 300건
[배당요구] 300건
[배당이의] 300건
[인도명령] 300건
[건물명도] 300건
[임대인 수선의무] 120건
[원상회복] 300건
[누수] 300건
[하자] 300건
[통상의 손모] 3건
[공인중개사 책임] 300건
[중개대상물 확인설명] 93건
[중개보수] 300건

수집 9105건 → 판례일련번호 1차 dedup 6372건


In [6]:
# 2) 상세 조회 → 본문 유효성 필터 → 2차 dedup
prec_details = fetch_details("prec", prec_items, id_field="판례일련번호")
prec_details = dedup_by(prec_details, "판례일련번호")

# 사건번호 중복(같은 사건 다른 심급)은 로깅만, 제거는 전처리 단계 판단
case_nos = [str(r.get("사건번호", "")) for r in prec_details if r.get("사건번호")]
dup_cases = len(case_nos) - len(set(case_nos))
print(f"판례 최종 {len(prec_details)}건 (사건번호 중복 {dup_cases}건 — 전처리에서 판단)")

# 3) JSONL 저장
prec_path = RAW_DIR / "prec.jsonl"
with prec_path.open("w", encoding="utf-8") as f:
    for rec in prec_details:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"저장 완료 → {prec_path}")

  상세 조회 시작: 총 6372건 (건당 ~0.5s sleep)
    진행 40/6372 — 유효 33 / 제거 7
    진행 60/6372 — 유효 51 / 제거 9
    진행 100/6372 — 유효 81 / 제거 19
    진행 140/6372 — 유효 107 / 제거 33
    진행 160/6372 — 유효 116 / 제거 44
    진행 180/6372 — 유효 127 / 제거 53
    진행 220/6372 — 유효 146 / 제거 74
    진행 260/6372 — 유효 161 / 제거 99
    진행 280/6372 — 유효 170 / 제거 110
    진행 320/6372 — 유효 197 / 제거 123
    진행 340/6372 — 유효 213 / 제거 127
    진행 400/6372 — 유효 239 / 제거 161
    진행 420/6372 — 유효 250 / 제거 170
    진행 480/6372 — 유효 282 / 제거 198
    진행 520/6372 — 유효 306 / 제거 214
    진행 560/6372 — 유효 331 / 제거 229
    진행 580/6372 — 유효 341 / 제거 239
    진행 600/6372 — 유효 352 / 제거 248
    진행 620/6372 — 유효 365 / 제거 255
    진행 640/6372 — 유효 372 / 제거 268
    진행 680/6372 — 유효 391 / 제거 289
    진행 720/6372 — 유효 409 / 제거 311
    진행 740/6372 — 유효 422 / 제거 318
    진행 760/6372 — 유효 430 / 제거 330
    진행 840/6372 — 유효 466 / 제거 374
    진행 880/6372 — 유효 479 / 제거 401
    진행 920/6372 — 유효 498 / 제거 422
    진행 940/6372 — 유효 505 / 제거 435
    진행 960/6372 — 유효 517 /

HTTPError: 404 Client Error: Not Found for url: https://www.law.go.kr/DRF/lawService.do?OC=jungia21&type=JSON&target=prec&ID=230022

## 3. 법령해석례 (expc)

조문·개념 중심 키워드로 본문검색(`search=2`) → `법령해석례일련번호` dedup → 상세 조회 →
2차 검증 → `expc.jsonl` 저장.

In [ ]:
EXPC_QUERIES: list[str] = [
    "주택임대차보호법", "임대차보증금 우선변제", "대항력", "확정일자", "계약갱신청구권",
    "묵시적 갱신", "차임 증액", "임차권등기명령", "소액임차인 최우선변제", "전입신고",
    "상가건물 임대차",
]

# 1) 본문검색 → 합치기 → 법령해석례일련번호 1차 dedup
expc_items: list[dict] = []
for q in EXPC_QUERIES:
    hits = fetch_list("expc", query=q, search=2)
    print(f"[{q}] {len(hits)}건")
    expc_items.extend(hits)
    time.sleep(0.3)

before = len(expc_items)
expc_items = dedup_by(expc_items, "법령해석례일련번호")
print(f"\n수집 {before}건 → 법령해석례일련번호 1차 dedup {len(expc_items)}건")

# 2) 상세 조회 → 본문 유효성 필터 → 2차 dedup
expc_details = fetch_details("expc", expc_items, id_field="법령해석례일련번호")
expc_details = dedup_by(expc_details, "법령해석례일련번호")

# 3) JSONL 저장
expc_path = RAW_DIR / "expc.jsonl"
with expc_path.open("w", encoding="utf-8") as f:
    for rec in expc_details:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"저장 완료: {len(expc_details)}건 → {expc_path}")

## 4. 수집 결과 요약 / 검증

In [ ]:
import pandas as pd

# 법령: 파일 수
eflaw_files = sorted(EFLAW_DIR.glob("*.json"))
print(f"eflaw: {len(eflaw_files)}개 파일 → {EFLAW_DIR}")

# 판례/해석례: 건수 + ID 유일성 + OC 키 미노출 검증
for target, idf in [("prec", "판례일련번호"), ("expc", "법령해석례일련번호")]:
    path = RAW_DIR / f"{target}.jsonl"
    if not path.exists():
        print(f"{target}: 파일 없음")
        continue
    df = pd.read_json(path, lines=True)
    ids = df[idf].astype(str).tolist()
    leak = df.astype(str).apply(lambda c: c.str.contains("OC=", na=False)).any().any()
    print(
        f"{target}: {len(df)}건 | ID 유일 {len(ids) == len(set(ids))} | OC 키 노출 {leak}"
    )